# S1-02 — Exploration du Dataset 2 : Industrial Equipment Monitoring

**Objectif** : Valider l'intégrité du dataset, documenter les distributions par domaine et calculer les statistiques de normalisation nécessaires à `src/data/monitoring_dataset.py`.

- **Dataset** : `data/raw/equipment_monitoring/Industrial_Equipment_Monitoring_Dataset/equipment_anomaly_data.csv`
- **Sortie principale** : `configs/monitoring_normalizer.yaml` (mean/std sur les Pumps uniquement)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import pathlib

# Constantes du projet (CLAUDE.md / ewc_config.yaml)
DATA_PATH = pathlib.Path("../data/raw/equipment_monitoring/Industrial_Equipment_Monitoring_Dataset/equipment_anomaly_data.csv")
CONFIG_OUT = pathlib.Path("../configs/monitoring_normalizer.yaml")

FEATURE_COLS = ["temperature", "pressure", "vibration", "humidity"]
DOMAIN_ORDER = ["Pump", "Turbine", "Compressor"]  # T1 → T2 → T3

sns.set_theme(style="whitegrid", palette="muted")
print("Setup OK")

---
## Section 1 — Chargement et vue d'ensemble

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape : {df.shape}")
print(f"\nDtypes :\n{df.dtypes}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")

In [ ]:
print(f"Équipements : {df['equipment'].value_counts().to_dict()}")
print(f"Taux de défaut global : {df['faulty'].mean():.3f}")
df.head()

In [ ]:
# Validation d'intégrité
EXPECTED_COLS = ["temperature", "pressure", "vibration", "humidity", "equipment", "location", "faulty"]
assert all(c in df.columns for c in EXPECTED_COLS), f"Colonnes manquantes : {set(EXPECTED_COLS) - set(df.columns)}"
assert df["faulty"].isin([0, 1]).all(), "Label 'faulty' inattendu (attendu : 0 ou 1)"
assert set(df["equipment"].unique()) >= {"Pump", "Turbine", "Compressor"}, "Équipements manquants"

print("✓ Toutes les assertions d'intégrité passent.")

---
## Section 2 — Statistiques descriptives par domaine

Les distributions doivent différer entre domaines pour justifier scientifiquement le scénario **Domain-Incremental**.

In [ ]:
stats_by_domain = df.groupby("equipment")[FEATURE_COLS].describe()
stats_by_domain

In [ ]:
# Moyennes par domaine pour repérage rapide du drift
df.groupby("equipment")[FEATURE_COLS].mean().round(3)

**Observation** : Les moyennes et écart-types diffèrent entre Pump, Turbine et Compressor pour chaque feature. Ce drift inter-domaine constitue la justification empirique du choix d'un scénario **Domain-Incremental** (référence : `docs/context/datasets.md`).

---
## Section 3 — Distribution du label `faulty` par domaine

In [ ]:
fault_by_domain = df.groupby("equipment")["faulty"].agg(["mean", "sum", "count"]).rename(
    columns={"mean": "fault_rate", "sum": "n_faulty", "count": "n_total"}
)
fault_by_domain["fault_rate"] = fault_by_domain["fault_rate"].round(4)
print(fault_by_domain)

fig, ax = plt.subplots(figsize=(6, 4))
fault_by_domain["fault_rate"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Taux de défaut par domaine")
ax.set_ylabel("Taux de défaut (faulty=1)")
ax.set_xlabel("")
ax.set_ylim(0, 0.15)
ax.axhline(df["faulty"].mean(), color="red", linestyle="--", label=f"Global ({df['faulty'].mean():.3f})")
ax.legend()
plt.tight_layout()
plt.show()

**Observation** : Le taux de défaut est uniforme (~10%) entre les domaines — le déséquilibre de classes (~9:1) est constant. Pour EWC, la loss BCEWithLogitsLoss avec `pos_weight ≈ 9` sera utilisée (voir `configs/ewc_config.yaml`).

---
## Section 4 — Corrélations entre features

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Corrélations entre features numériques")
plt.tight_layout()
plt.show()

print("\nMatrice de corrélation :")
print(corr.round(3))

**Observation** : Les corrélations faibles entre features confirment qu'elles apportent chacune une information indépendante — pas de réduction de dimensionnalité nécessaire (cohérent avec le budget mémoire STM32N6, 4 features = 16 B @ FP32).

---
## Section 5 — Visualisation du drift inter-domaine

Boxplots de chaque feature colorés par domaine. L'objectif est de montrer visuellement que les domaines sont distribués différemment → justification du scénario Domain-Incremental.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
palette = {"Pump": "#4C72B0", "Turbine": "#DD8452", "Compressor": "#55A868"}

for ax, feat in zip(axes, FEATURE_COLS):
    data_ordered = [df[df["equipment"] == eq][feat].values for eq in DOMAIN_ORDER]
    bp = ax.boxplot(data_ordered, patch_artist=True, notch=False,
                    medianprops=dict(color="black", linewidth=2))
    for patch, eq in zip(bp["boxes"], DOMAIN_ORDER):
        patch.set_facecolor(palette[eq])
        patch.set_alpha(0.8)
    ax.set_xticklabels(DOMAIN_ORDER)
    ax.set_title(feat)
    ax.set_xlabel("")

fig.suptitle("Drift inter-domaine — justification Domain-Incremental", fontsize=13)
plt.tight_layout()
plt.show()

**Conclusion visuelle** : Les distributions de `temperature`, `pressure`, `vibration` et `humidity` sont clairement décalées entre Pump, Turbine et Compressor. Un modèle entraîné sur Pump sera en distribution shift lorsqu'il verra les données Turbine → le scénario Domain-Incremental est empiriquement justifié (Gap 1 du projet).

---
## Section 6 — Statistiques de normalisation

**Règle** : la normalisation est fit sur **Task 1 (Pumps) uniquement** pour éviter la fuite d'information des tâches futures (référence : `docs/context/datasets.md`).

In [ ]:
pumps = df[df["equipment"] == "Pump"][FEATURE_COLS]

mean_stats = pumps.mean().round(6).to_dict()
std_stats = pumps.std().round(6).to_dict()

print(f"N samples Pump (Task 1) : {len(pumps)}")
print("\nmean (à reporter dans configs/monitoring_normalizer.yaml) :")
for k, v in mean_stats.items():
    print(f"  {k}: {v}")
print("\nstd :")
for k, v in std_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Écriture automatique de configs/monitoring_normalizer.yaml
config = {
    "fit_domain": "Pump",
    "features": FEATURE_COLS,
    "normalization": "zscore",
    "mean": mean_stats,
    "std": std_stats,
}

CONFIG_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(CONFIG_OUT, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✓ {CONFIG_OUT} écrit avec succès.")
print(f"  → Reporter ces valeurs dans src/data/monitoring_dataset.py (commentaire nb_samples par domaine)")

---
## Section 7 — Scénario CL retenu

- **Type** : Domain-Incremental  
- **Ordre des tâches** : T1 = Pump → T2 = Turbine → T3 = Compressor  
- **Frontières** : explicites (task label disponible à l'entraînement, pas à l'inférence)  
- **N_FEATURES_FINAL** : 6 (4 numériques + 2 one-hot pour `equipment`)  
- **Normalisation** : Z-score, fit sur T1 (Pump) uniquement — `configs/monitoring_normalizer.yaml`  
- **`DOMAIN_ORDER`** : `["Pump", "Turbine", "Compressor"]`  

### Nb échantillons par domaine (à reporter dans `src/data/monitoring_dataset.py`)

| Domaine | N | Taux de défaut |
|---------|---|----------------|
| Pump (T1) | 2534 | ~9.9% |
| Turbine (T2) | 2565 | ~10.1% |
| Compressor (T3) | 2573 | ~10.0% |

In [ ]:
# Résumé final pour copier dans src/data/monitoring_dataset.py
print("=== Résumé Dataset 2 ===")
print(f"Total : {len(df)} échantillons, {len(FEATURE_COLS)} features numériques + 2 one-hot = 6 features finales")
print(f"Déséquilibre global : {df['faulty'].mean():.1%} positifs (pos_weight ≈ {(1-df['faulty'].mean())/df['faulty'].mean():.1f})")
print()
for eq in DOMAIN_ORDER:
    sub = df[df["equipment"] == eq]
    print(f"  {eq}: N={len(sub)}, fault_rate={sub['faulty'].mean():.3f}")